In [7]:
import trimesh
import pymesh
import os
import glob
import logging

def setup_logging():
    # Set up the logging configuration
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.StreamHandler(),  # Send log messages to standard output
            logging.FileHandler("remeshing_log.txt")  # Send log messages to a file
        ]
    )

def process_mesh(mesh, use_pymesh):
    if use_pymesh:
        logging.info("Using PyMesh for remeshing...")
        # Convert Trimesh mesh to PyMesh mesh
        pymesh_mesh = pymesh.form_mesh(mesh.vertices, mesh.faces)

        # Perform remeshing with a control over the triangle quality
        pymesh_mesh, _ = pymesh.remesh(pymesh_mesh, 0.1)

        # Convert back to Trimesh
        mesh = trimesh.Trimesh(vertices=pymesh_mesh.vertices, faces=pymesh_mesh.faces)
        logging.info("Remeshing with PyMesh completed.")
    else:
        logging.info("Using Trimesh for triangulation...")

        # Ensure the mesh is watertight (optional, depending on your needs)
        if not mesh.is_watertight:
            logging.info("Filling holes to make the mesh watertight.")
            success = mesh.fill_holes()
            if success:
                logging.info("Holes filled successfully.")
            else:
                logging.warning("Failed to fill holes.")

        # Fix the normals of the mesh to ensure they are consistent
        if hasattr(mesh, 'fix_normals'):
            mesh.fix_normals()
        else:
            logging.warning("Cannot fix normals on this mesh. Skipping.")

        # If any faces are not triangles, subdivide them
        if mesh.faces.shape[1] != 3:
            mesh = mesh.subdivide()

        logging.info("Triangulation with Trimesh completed.")
    
    return mesh

def remesh_to_triangles(input_folder, suffix, use_pymesh=False):
    setup_logging()

    # Convert the input path to an absolute path if it's not already
    input_folder = os.path.abspath(input_folder)

    logging.info("Starting remeshing process...")
    logging.info(f"Input folder: {input_folder}")
    logging.info(f"Using PyMesh: {use_pymesh}")

    # Find all .obj files recursively in the input_folder
    obj_files = glob.glob(os.path.join(input_folder, '**', '*.obj'), recursive=True)
    
    if not obj_files:
        logging.warning("No .obj files found in the specified input folder.")
        return

    logging.info(f"Found {len(obj_files)} .obj files to process.")

    for obj_file in obj_files:
        logging.info(f"Processing file: {obj_file}")

        try:
            # Load the object using Trimesh
            loaded = trimesh.load(obj_file, process=False)
            meshes = []

            if isinstance(loaded, trimesh.Trimesh):
                # Single mesh
                meshes = [loaded]
            elif isinstance(loaded, trimesh.Scene):
                # Multiple meshes in the scene
                meshes = [trimesh.Trimesh(vertices=g.vertices, faces=g.faces) for g in loaded.geometry.values()]
            
            for i, mesh in enumerate(meshes):
                processed_mesh = process_mesh(mesh, use_pymesh)

                # Generate the output file path by adding the suffix before the file extension
                file_root, file_ext = os.path.splitext(obj_file)
                output_file = f"{file_root}{suffix}_part{i}{file_ext}" if len(meshes) > 1 else f"{file_root}{suffix}{file_ext}"

                # Export the mesh back to .obj format
                processed_mesh.export(output_file, file_type='obj')
                logging.info(f"Remeshed file saved as: {output_file}")

        except Exception as e:
            logging.error(f"Error processing {obj_file}: {e}")

    logging.info("Remeshing process completed.")

# Define the input folder
input_folder = 'datasets/BIM_IKEA'

# Set PyMesh usage
use_pymesh = True

# Set the suffix based on the tool used
suffix = 'tri_pymesh' if use_pymesh else 'tri_trimesh'

# Run the remeshing process
remesh_to_triangles(input_folder, suffix, use_pymesh=use_pymesh)


2024-08-18 11:48:21,630 - INFO - Starting remeshing process...
2024-08-18 11:48:21,631 - INFO - Input folder: /Users/matteo/Desktop/Teo/Projects/meshgpt-pytorch/datasets/BIM_IKEA
2024-08-18 11:48:21,631 - INFO - Using PyMesh: False
2024-08-18 11:48:21,636 - INFO - Found 56 .obj files to process.
2024-08-18 11:48:21,636 - INFO - Processing file: /Users/matteo/Desktop/Teo/Projects/meshgpt-pytorch/datasets/BIM_IKEA/IKE040015_obj/IKEA-Liatrop_Coffe_Table-3D.obj
2024-08-18 11:48:21,661 - INFO - triangulating faces
2024-08-18 11:48:21,662 - INFO - triangulating faces
2024-08-18 11:48:21,665 - INFO - Using Trimesh for triangulation...
2024-08-18 11:48:21,666 - INFO - Triangulation with Trimesh completed.
2024-08-18 11:48:21,667 - INFO - Remeshed file saved as: /Users/matteo/Desktop/Teo/Projects/meshgpt-pytorch/datasets/BIM_IKEA/IKE040015_obj/IKEA-Liatrop_Coffe_Table-3Dtri_trimesh_part0.obj
2024-08-18 11:48:21,667 - INFO - Using Trimesh for triangulation...
2024-08-18 11:48:21,671 - INFO - Fil